In [ ]:
!pip install -q nibabel trimesh fast-simplification plotly

# MARCHING CUBES

In [ ]:
import os
import numpy as np
import nibabel as nib
import trimesh
import plotly.graph_objects as go
from skimage import measure
from scipy import ndimage
from scipy.ndimage import binary_closing, binary_opening
import warnings
warnings.filterwarnings('ignore')

print('Libraries ready')

In [ ]:
NII_PATH   = "/kaggle/input/datasets/khoayakult/stestdata/case_00000_merged.nii"
OUTPUT_DIR = "/kaggle/working"
HTML_OUT   = os.path.join(OUTPUT_DIR, "3d_kidney_reconstruction_00066_khoa_1.html")

# Kiem tra file ton tai
assert os.path.exists(NII_PATH), f"Khong tim thay file: {NII_PATH}"
print(f"File found: {NII_PATH}")
print(f"File size : {os.path.getsize(NII_PATH)/1e6:.1f} MB")

In [ ]:
print("Loading NIfTI file...")
nii     = nib.load(NII_PATH)
seg_raw = nii.get_fdata(dtype=np.float32)   # [X, Y, Z]
spacing = np.array(nii.header.get_zooms())  # (sx, sy, sz) mm/voxel

print(f"Volume shape : {seg_raw.shape}")
print(f"Voxel spacing: {spacing} mm")
print(f"Unique labels: {np.unique(seg_raw.astype(int)).tolist()}")

# Phan phoi nhan
label_map = {0: 'Background', 1: 'Kidney', 2: 'Tumor/Mass', 
             3: 'Cyst', 4: 'Label4', 5: 'Label5'}
for lab in np.unique(seg_raw.astype(int)):
    count = int((seg_raw == lab).sum())
    pct   = count / seg_raw.size * 100
    name  = label_map.get(lab, f'Unknown({lab})')
    print(f"  Label {lab} ({name}): {count:,} voxels ({pct:.2f}%)")

segmentation = seg_raw.astype(np.uint8)

In [ ]:
class EnhancedMarchingCubes:
    """
    Pipeline hoan chinh:
      preprocess -> extract_surface -> smooth_taubin -> decimate -> remove_small_components
    """

    def __init__(self, volume, spacing=(1.0, 1.0, 1.0)):
        self.volume  = volume.astype(float)
        self.spacing = np.array(spacing, dtype=float)
        self.mesh    = None

    # ----------------------------------------------------------
    def preprocess_volume(self, sigma=0.6, use_morphology=True):
        """
        Gaussian blur de lam mo canh + morphological closing lap lo nho.
        sigma cao hon = mesh muot hon nhung mat chi tiet nho.
        """
        vol = self.volume.copy()

        if use_morphology:
            binary = vol > 0.5
            # Closing: lap cac lo nho ben trong
            binary = binary_closing(binary, structure=np.ones((3, 3, 3)))
            # Opening: xoa cac diem nhieu nho ben ngoai
            binary = binary_opening(binary, structure=np.ones((2, 2, 2)))
            vol    = binary.astype(float)

        smoothed = ndimage.gaussian_filter(vol, sigma=sigma)
        return smoothed

    # ----------------------------------------------------------
    def extract_surface(self, level=0.5, step_size=1, sigma=0.6, use_morphology=True):
        smoothed = self.preprocess_volume(sigma=sigma, use_morphology=use_morphology)
    
        if smoothed.max() < level:
            print("  -> Volume trang, bo qua.")
            return False
    
        try:
            verts, faces, normals, values = measure.marching_cubes(
                smoothed,
                level=level,
                spacing=tuple(self.spacing),
                step_size=step_size,
                allow_degenerate=False,
            )
            # Fix winding de volume duong
            tm = trimesh.Trimesh(vertices=verts, faces=faces, process=False)
            trimesh.repair.fix_winding(tm)
            trimesh.repair.fix_normals(tm)
    
            self.mesh = {
                'vertices': np.array(tm.vertices),
                'faces':    np.array(tm.faces),
                'normals':  normals,
                'values':   values,
            }
            print(f"  Extracted: {len(tm.vertices):,} verts | {len(tm.faces):,} faces")
            return True
        except Exception as e:
            print(f"  Marching Cubes ERROR: {e}")
            return False

    # ----------------------------------------------------------
    def decimate(self, reduction_ratio=0.3):
        """
        Giam so luong tam giac xuong con reduction_ratio * so cu.
        reduction_ratio=0.3 -> giu lai 30% tam giac.
        """
        if self.mesh is None:
            return

        n_original = len(self.mesh['faces'])
        target     = max(500, int(n_original * reduction_ratio))

        try:
            tm = trimesh.Trimesh(
                vertices=self.mesh['vertices'],
                faces=self.mesh['faces'],
                process=False,
            )
            simplified = tm.simplify_quadric_decimation(target)
            self.mesh['vertices'] = np.array(simplified.vertices)
            self.mesh['faces']    = np.array(simplified.faces)
        except Exception as e:
            print(f"  Decimate failed (kept original): {e}")

    # ----------------------------------------------------------
    def remove_small_components(self, min_faces=50):
        """Giu lai cac connected components co it nhat min_faces tam giac."""
        if self.mesh is None:
            return

        tm = trimesh.Trimesh(
            vertices=self.mesh['vertices'],
            faces=self.mesh['faces'],
            process=False,
        )
        components = tm.split(only_watertight=False)
        large = [c for c in components if len(c.faces) >= min_faces]

        if not large:
            print(f"  WARNING: Khong con component nao >= {min_faces} faces sau loc.")
            return

        merged = trimesh.util.concatenate(large)
        self.mesh['vertices'] = np.array(merged.vertices)
        self.mesh['faces']    = np.array(merged.faces)
        print(f"  Components: {len(components)} total, giu {len(large)} | {len(merged.faces):,} faces")

    
    def smooth_taubin(self, iterations=15, lambda_val=0.5, mu_val=-0.53):
        """
        Taubin smoothing: khong gay shrinkage nhu Laplacian.
        Xen ke buoc lambda (shrink) va mu (inflate) de giu the tich.
        """
        if self.mesh is None:
            return
    
        vertices = self.mesh['vertices'].copy()
        faces    = self.mesh['faces']
        n        = len(vertices)
    
        # Xay dung danh sach ke
        neighbors = [[] for _ in range(n)]
        for f in faces:
            for i in range(3):
                a, b = f[i], f[(i + 1) % 3]
                neighbors[a].append(b)
                neighbors[b].append(a)
        # Loai trung lap
        neighbors = [list(set(nb)) for nb in neighbors]
    
        def _step(verts, factor):
            new_v = verts.copy()
            for i in range(n):
                nb = neighbors[i]
                if nb:
                    centroid = verts[nb].mean(axis=0)
                    new_v[i] = verts[i] + factor * (centroid - verts[i])
            return new_v
    
        for _ in range(iterations):
            vertices = _step(vertices, lambda_val)
            vertices = _step(vertices, mu_val)
    
        self.mesh['vertices'] = vertices
        print(f"  Taubin smoothing done ({iterations} iter)")
    
    # ----------------------------------------------------------
    def get_stats(self):
        if self.mesh is None:
            return {}
        tm = trimesh.Trimesh(
            vertices=self.mesh['vertices'],
            faces=self.mesh['faces'],
            process=False,
        )
        return {
            'vertices':     len(self.mesh['vertices']),
            'faces':        len(self.mesh['faces']),
            'surface_area': round(float(tm.area), 2),
            'volume_mm3':   round(float(tm.volume), 2) if tm.is_watertight else None,
            'is_watertight': tm.is_watertight,
        }

print("EnhancedMarchingCubes class ready")

In [ ]:
print("=" * 60)
print("BUOC 1: Auto-crop de tiet kiem RAM")
print("=" * 60)

# Chi crop theo vung co nhan > 0
fg = np.where(segmentation > 0)
if len(fg[0]) == 0:
    raise ValueError("Segmentation trong rong! Kiem tra lai file NIfTI.")

MARGIN = 10
x0 = max(0, fg[0].min() - MARGIN);  x1 = min(segmentation.shape[0], fg[0].max() + MARGIN)
y0 = max(0, fg[1].min() - MARGIN);  y1 = min(segmentation.shape[1], fg[1].max() + MARGIN)
z0 = max(0, fg[2].min() - MARGIN);  z1 = min(segmentation.shape[2], fg[2].max() + MARGIN)

cropped = segmentation[x0:x1, y0:y1, z0:z1]
print(f"Shape goc : {segmentation.shape}")
print(f"Shape crop: {cropped.shape}")
print(f"Spacing   : {spacing} mm")

# ============================================================
print("\n" + "=" * 60)
print("BUOC 2: Xay dung mesh cho tung bo phan")
print("=" * 60)

PARTS = [
    {'label': 1, 'name': 'Than (Kidney)',  'color': '#E8735A', 'opacity': 0.35,
     'sigma': 0.8, 'smooth_iter': 15, 'min_faces': 200},

    {'label': 2, 'name': 'Khoi U (Tumor)', 'color': '#FFD700', 'opacity': 0.95,
     'sigma': 0.5, 'smooth_iter': 10, 'min_faces': 30},

    {'label': 3, 'name': 'Nang (Cyst)',    'color': '#00CED1', 'opacity': 0.90,
     'sigma': 0.5, 'smooth_iter': 10, 'min_faces': 30},

    {'label': 4, 'name': 'Label 4',        'color': '#AA00FF', 'opacity': 0.90,
     'sigma': 0.5, 'smooth_iter': 10, 'min_faces': 20},

    {'label': 5, 'name': 'Label 5',        'color': '#00FF88', 'opacity': 0.90,
     'sigma': 0.5, 'smooth_iter': 10, 'min_faces': 20},
]

fig     = go.Figure()
summary = []

for part in PARTS:
    lab  = part['label']
    name = part['name']
    print(f"\n--- {name} (label={lab}) ---")

    mask = (cropped == lab).astype(np.uint8)
    n_vox = int(mask.sum())

    if n_vox == 0:
        print(f"  Khong tim thay {name} trong ca benh nay. Bo qua.")
        continue

    print(f"  Voxels: {n_vox:,}")

    mc = EnhancedMarchingCubes(mask.astype(float), spacing=spacing)

    if not mc.extract_surface(
        level=0.5,
        step_size=1,
        sigma=part['sigma'],
        use_morphology=True,
    ):
        continue

    mc.smooth_taubin(iterations=part['smooth_iter'])
    mc.remove_small_components(min_faces=part['min_faces'])

    stats = mc.get_stats()
    stats['name'] = name
    summary.append(stats)
    print(f"  Stats: {stats}")

    verts = mc.mesh['vertices']
    faces = mc.mesh['faces']

    fig.add_trace(go.Mesh3d(
        x=verts[:, 0], y=verts[:, 1], z=verts[:, 2],
        i=faces[:, 0], j=faces[:, 1], k=faces[:, 2],
        color=part['color'],
        opacity=part['opacity'],
        name=name,
        showscale=False,
        flatshading=False,
        # Lighting de trong nhu nhua y sinh
        lighting=dict(
            ambient=0.4,
            diffuse=0.8,
            specular=0.3,
            roughness=0.5,
            fresnel=0.2,
        ),
        lightposition=dict(x=100, y=200, z=300),
    ))
    print(f"  Da them {name} vao mo hinh 3D.")

print("\n" + "=" * 60)
print("Tat ca bo phan da duoc xu ly xong.")

In [ ]:
case_name = os.path.basename(NII_PATH).replace('.nii.gz', '').replace('.nii', '')

fig.update_layout(
    title=dict(
        text=f'Mo hinh 3D: Than - Khoi U - Nang | {case_name}',
        x=0.5,
        font=dict(size=16),
    ),
    scene=dict(
        xaxis=dict(title='X (mm)', showbackground=True, backgroundcolor='#f0f0f0'),
        yaxis=dict(title='Y (mm)', showbackground=True, backgroundcolor='#e8e8e8'),
        zaxis=dict(title='Z (mm)', showbackground=True, backgroundcolor='#f0f0f0'),
        aspectmode='data',        # Giu dung ti le vat ly, khong bi meo
        bgcolor='#1a1a2e',        # Nen toi de biet ro cac bo phan
        camera=dict(
            eye=dict(x=1.5, y=1.5, z=1.2)  # Goc nhin mac dinh hop ly
        ),
    ),
    paper_bgcolor='#1a1a2e',
    font=dict(color='white'),
    legend=dict(
        x=0.01, y=0.99,
        bgcolor='rgba(0,0,0,0.5)',
        font=dict(color='white', size=12),
    ),
    width=1000,
    height=850,
    margin=dict(r=10, l=10, b=10, t=60),
)

fig.write_html(
    HTML_OUT,
    include_plotlyjs='cdn',   # Nhe hon: chi load plotly.js tu CDN thay vi nhung vao file
    full_html=True,
)

size_mb = os.path.getsize(HTML_OUT) / 1e6
print(f"HTML saved : {HTML_OUT}")
print(f"File size  : {size_mb:.1f} MB")

# MONAI + VTK

In [1]:
!pip install monai pyvista
!git clone https://github.com/Project-MONAI/MONAI

fatal: destination path 'MONAI' already exists and is not an empty directory.


In [2]:
!pip install -q monai pyvista nibabel trimesh plotly

In [7]:
import os
import numpy as np
import nibabel as nib
import pyvista as pv
import trimesh
import plotly.graph_objects as go
import warnings

from monai.transforms import (
    LoadImage,
    EnsureChannelFirst,
    Spacing,
)

warnings.filterwarnings('ignore')

# PyVista chạy headless trên Kaggle (không cần display)
pv.global_theme.jupyter_backend = 'none'

In [11]:
import os
NII_PATH   = "/kaggle/input/datasets/khoayakult/stestdata/case_00000_merged.nii"
OUTPUT_DIR = "/kaggle/working"

case_name  = os.path.basename(NII_PATH).replace('.nii.gz','').replace('.nii','')
HTML_OUT   = os.path.join(OUTPUT_DIR, f"3d_{case_name}_monai_pyvista.html")
GLB_OUT    = os.path.join(OUTPUT_DIR, f"3d_{case_name}_monai_pyvista.glb")

In [12]:
# ============================================================
# CELL 4 — Load NIfTI + MONAI Isotropic Resampling
#
# TẠI SAO CẦN BƯỚC NÀY?
#   CT scan thường có spacing ví dụ [0.78, 0.78, 3.0] mm
#   → trục Z thưa hơn 4x so với X,Y
#   → nếu chạy MC trực tiếp, mesh bị "bẹp" theo Z
#   MONAI Spacing resample về 1×1×1 mm trước khi extract surface
# ============================================================
print('Loading NIfTI file...')
nii         = nib.load(NII_PATH)
spacing_orig = np.array(nii.header.get_zooms())
print(f"  Original shape  : {nii.get_fdata(dtype=np.float32).shape}")
print(f"  Original spacing: {spacing_orig} mm")

print('\nRunning MONAI isotropic resampling...')
loader = LoadImage(image_only=True)
img    = loader(NII_PATH)
img    = EnsureChannelFirst()(img)           # [C, X, Y, Z]

# Resample về 1×1×1 mm, mode='nearest' giữ đúng label integer
img_iso = Spacing(pixdim=(1.0, 1.0, 1.0), mode='nearest')(img)
seg_iso = img_iso[0].numpy().astype(np.uint8)   # [X, Y, Z]

print(f"  Isotropic shape : {seg_iso.shape}")
print(f"  Unique labels   : {np.unique(seg_iso).tolist()}")

# Phân phối nhãn
label_map = {0:'Background', 1:'Kidney', 2:'Tumor/Mass', 3:'Cyst', 4:'Label4', 5:'Label5'}
for lab in np.unique(seg_iso):
    cnt = int((seg_iso == lab).sum())
    pct = cnt / seg_iso.size * 100
    print(f"  Label {lab} ({label_map.get(lab, '?')}): {cnt:,} voxels ({pct:.2f}%)")

Loading NIfTI file...
  Original shape  : (611, 512, 512)
  Original spacing: [0.5       0.9199219 0.9199219] mm

Running MONAI isotropic resampling...
  Isotropic shape : (306, 471, 471)
  Unique labels   : [0, 1, 2, 4, 5]
  Label 0 (Background): 67,513,586 voxels (99.46%)
  Label 1 (Kidney): 359,003 voxels (0.53%)
  Label 2 (Tumor/Mass): 5,092 voxels (0.01%)
  Label 4 (Label4): 1,858 voxels (0.00%)
  Label 5 (Label5): 3,807 voxels (0.01%)


In [13]:
# ============================================================
# CELL 5 — Auto-crop để tiết kiệm RAM
# ============================================================
fg = np.where(seg_iso > 0)
if len(fg[0]) == 0:
    raise ValueError('❌ Segmentation rỗng! Kiểm tra lại file.')

MARGIN = 15   # voxels — tăng lên 15 vì spacing đã 1mm, cần margin lớn hơn
x0 = max(0, fg[0].min()-MARGIN);  x1 = min(seg_iso.shape[0], fg[0].max()+MARGIN)
y0 = max(0, fg[1].min()-MARGIN);  y1 = min(seg_iso.shape[1], fg[1].max()+MARGIN)
z0 = max(0, fg[2].min()-MARGIN);  z1 = min(seg_iso.shape[2], fg[2].max()+MARGIN)

cropped = seg_iso[x0:x1, y0:y1, z0:z1]
print(f'Shape sau isotropic: {seg_iso.shape}')
print(f'Shape sau crop     : {cropped.shape}')
print(f'RAM usage ước tính : {cropped.nbytes / 1e6:.1f} MB')

Shape sau isotropic: (306, 471, 471)
Shape sau crop     : (161, 121, 238)
RAM usage ước tính : 4.6 MB


In [14]:
# ============================================================
# CELL 6 — Hàm tiện ích: PyVista mesh → Plotly Mesh3d
# ============================================================
def pyvista_to_plotly(pv_mesh, color, opacity, name):
    """
    Chuyển PyVista PolyData sang Plotly Mesh3d trace.
    PyVista lưu faces dạng [3, i, j, k, 3, i2, j2, k2, ...]
    → reshape (-1, 4) rồi bỏ cột đầu (số 3).
    """
    verts = np.array(pv_mesh.points)
    
    # Đảm bảo toàn bộ là tam giác
    tri_mesh = pv_mesh.triangulate()
    faces_flat = tri_mesh.faces.reshape(-1, 4)[:, 1:]  # (N, 3)
    
    return go.Mesh3d(
        x=verts[:, 0], y=verts[:, 1], z=verts[:, 2],
        i=faces_flat[:, 0], j=faces_flat[:, 1], k=faces_flat[:, 2],
        color=color,
        opacity=opacity,
        name=name,
        showscale=False,
        flatshading=False,
        lighting=dict(
            ambient=0.45,
            diffuse=0.85,
            specular=0.35,
            roughness=0.45,
            fresnel=0.25,
        ),
        lightposition=dict(x=100, y=200, z=300),
    )


def mesh_stats(pv_mesh, name):
    """In và trả về stats của mesh."""
    is_mani = pv_mesh.is_manifold
    vol     = round(float(pv_mesh.volume), 2) if is_mani else None
    sa      = round(float(pv_mesh.area), 2)
    stats = {
        'name': name,
        'vertices': pv_mesh.n_points,
        'faces': pv_mesh.n_faces,
        'surface_area_mm2': sa,
        'volume_mm3': vol,
        'is_manifold': is_mani,
    }
    print(f"  Stats → {stats}")
    return stats


print('✅ Utility functions ready')

✅ Utility functions ready


In [16]:
# ============================================================
# CELL 7 — Build mesh từng label bằng PyVista + VTK
#
# PIPELINE MỖI LABEL:
#   1. Tạo PyVista ImageData (point data)
#   2. contour([0.5])  → VTK Marching Cubes (C++)
#   3. smooth_taubin() → VTK Taubin smooth (C++)
#   4. decimate()      → VTK Quadric decimation
#   5. Lọc connected components nhỏ
# ============================================================
PARTS = [
    # label, name, color hex, opacity, smooth_iter, pass_band, decimate_ratio, min_faces
    # pass_band: nhỏ hơn = mượt hơn (0.01 rất mượt, 0.1 vừa phải)
    # decimate : 0.7 = xóa 70% tam giác, giữ 30%
    {'label': 1, 'name': 'Than (Kidney)',  'color': '#E8735A', 'opacity': 0.35,
     'smooth_iter': 30, 'pass_band': 0.05, 'decimate': 0.70, 'min_faces': 200},

    {'label': 2, 'name': 'Khoi U (Tumor)', 'color': '#FFD700', 'opacity': 0.95,
     'smooth_iter': 20, 'pass_band': 0.08, 'decimate': 0.50, 'min_faces': 30},

    {'label': 3, 'name': 'Nang (Cyst)',    'color': '#00CED1', 'opacity': 0.90,
     'smooth_iter': 20, 'pass_band': 0.08, 'decimate': 0.50, 'min_faces': 30},

    {'label': 4, 'name': 'Label 4',        'color': '#AA00FF', 'opacity': 0.90,
     'smooth_iter': 15, 'pass_band': 0.10, 'decimate': 0.50, 'min_faces': 20},

    {'label': 5, 'name': 'Label 5',        'color': '#00FF88', 'opacity': 0.90,
     'smooth_iter': 15, 'pass_band': 0.10, 'decimate': 0.50, 'min_faces': 20},
]

fig          = go.Figure()
summary      = []
meshes_all   = []   # Lưu để export GLB
colors_all   = []   # Màu tương ứng cho GLB

for part in PARTS:
    lab  = part['label']
    name = part['name']
    print(f"\n{'='*55}")
    print(f"  {name}  (label={lab})")
    print(f"{'='*55}")

    mask  = (cropped == lab).astype(np.uint8)
    n_vox = int(mask.sum())

    if n_vox == 0:
        print(f"  ⚠️  Không có voxel nào cho {name}. Bỏ qua.")
        continue

    print(f"  Voxels: {n_vox:,}")

    # ----------------------------------------------------------
    # BƯỚC 1 — Tạo PyVista ImageData
    # Spacing đã là 1×1×1 mm sau MONAI → không cần điều chỉnh thêm
    # ----------------------------------------------------------
    grid = pv.ImageData()
    grid.dimensions = mask.shape               # (nx, ny, nz) — point data
    grid.spacing    = (1.0, 1.0, 1.0)         # mm/voxel, isotropic
    grid.origin     = (0.0, 0.0, 0.0)
    # Lưu vào point_data để contour() hoạt động đúng
    grid.point_data['values'] = mask.flatten(order='F').astype(np.float32)

    # ----------------------------------------------------------
    # BƯỚC 2 — VTK Marching Cubes
    # contour() gọi vtkContourFilter (MC bên dưới)
    # level=0.5 → biên giữa 0 và 1
    # ----------------------------------------------------------
    try:
        mesh = grid.contour([0.5], scalars='values')
    except Exception as e:
        print(f"  ❌ MC Error: {e}")
        continue

    if mesh.n_points == 0:
        print("  ⚠️  Mesh rỗng sau MC. Bỏ qua.")
        continue

    print(f"  [MC]      → {mesh.n_points:,} verts | {mesh.n_faces:,} faces")

    # ----------------------------------------------------------
    # BƯỚC 3 — VTK Taubin Smoothing
    # pass_band nhỏ = mượt hơn nhưng mất chi tiết nhỏ
    # normalize_coordinates=True → ổn định hơn cho shape bất thường
    # ----------------------------------------------------------
    mesh = mesh.smooth_taubin(
        n_iter=part['smooth_iter'],
        pass_band=part['pass_band'],
        normalize_coordinates=True,
        feature_smoothing=False,   # Giữ edge features
    )
    print(f"  [Taubin]  → {mesh.n_points:,} verts | {mesh.n_faces:,} faces")

    # ----------------------------------------------------------
    # BƯỚC 4 — VTK Quadric Decimation
    # decimate(0.7) = xóa 70%, giữ 30%
    # ----------------------------------------------------------
    mesh = mesh.decimate(part['decimate'])
    print(f"  [Decimate]→ {mesh.n_points:,} verts | {mesh.n_faces:,} faces")

    # ----------------------------------------------------------
    # BƯỚC 5 — Lọc connected components nhỏ  (PATCHED)
    # Dùng trimesh thay vì PyVista threshold để tránh UnstructuredGrid issue
    # ----------------------------------------------------------
    verts_np = np.array(mesh.triangulate().points)
    faces_np = mesh.triangulate().faces.reshape(-1, 4)[:, 1:]

    tm       = trimesh.Trimesh(vertices=verts_np, faces=faces_np, process=False)
    comps    = tm.split(only_watertight=False)

    large_parts = [c for c in comps if len(c.faces) >= part['min_faces']]

    if not large_parts:
        print(f"  ⚠️  Không có component nào >= {part['min_faces']} faces. Giữ lớn nhất.")
        large_parts = [max(comps, key=lambda c: len(c.faces))]

    merged = trimesh.util.concatenate(large_parts) if len(large_parts) > 1 else large_parts[0]

    # Convert trimesh → PyVista để dùng tiếp
    mesh = pv.wrap(merged)
    mesh = mesh.triangulate()
    print(f"  [Filter]  → {mesh.n_points:,} verts | {mesh.n_faces:,} faces | "
          f"{len(large_parts)}/{len(comps)} component(s) giữ lại")

    # ----------------------------------------------------------
    # Stats
    # ----------------------------------------------------------
    s = mesh_stats(mesh, name)
    summary.append(s)

    # Thêm vào Plotly figure
    fig.add_trace(pyvista_to_plotly(mesh, part['color'], part['opacity'], name))

    # Lưu để export GLB
    meshes_all.append(mesh)
    colors_all.append(part['color'])

    print(f"  ✅ Đã thêm {name} vào model 3D.")

print(f"\n{'='*55}")
print(f"Hoàn thành tất cả labels.")


  Than (Kidney)  (label=1)
  Voxels: 359,003
  [MC]      → 70,990 verts | 141,692 faces
  [Taubin]  → 70,990 verts | 141,692 faces
  [Decimate]→ 21,274 verts | 42,506 faces
  [Filter]  → 20,847 verts | 41,706 faces | 9/26 component(s) giữ lại
  Stats → {'name': 'Than (Kidney)', 'vertices': 20847, 'faces': 41706, 'surface_area_mm2': 43754.5, 'volume_mm3': 358161.57, 'is_manifold': True}
  ✅ Đã thêm Than (Kidney) vào model 3D.

  Khoi U (Tumor)  (label=2)
  Voxels: 5,092
  [MC]      → 5,018 verts | 10,012 faces
  [Taubin]  → 5,018 verts | 10,012 faces
  [Decimate]→ 2,490 verts | 5,005 faces
  [Filter]  → 2,490 verts | 5,004 faces | 2/3 component(s) giữ lại
  Stats → {'name': 'Khoi U (Tumor)', 'vertices': 2490, 'faces': 5004, 'surface_area_mm2': 3074.91, 'volume_mm3': 4918.81, 'is_manifold': True}
  ✅ Đã thêm Khoi U (Tumor) vào model 3D.

  Nang (Cyst)  (label=3)
  ⚠️  Không có voxel nào cho Nang (Cyst). Bỏ qua.

  Label 4  (label=4)
  Voxels: 1,858
  [MC]      → 4,836 verts | 9,292 face

In [17]:
# ============================================================
# CELL 8 — Summary bảng kết quả
# ============================================================
print('\n📊 SUMMARY')
print(f"{'─'*70}")
print(f"{'Tên':<20} {'Verts':>8} {'Faces':>8} {'SA (mm²)':>12} {'Vol (mm³)':>12} {'Watertight':>10}")
print(f"{'─'*70}")
for s in summary:
    vol_str = f"{s['volume_mm3']:,.0f}" if s['volume_mm3'] else 'N/A'
    print(f"{s['name']:<20} {s['vertices']:>8,} {s['faces']:>8,} "
          f"{s['surface_area_mm2']:>12,.1f} {vol_str:>12} {str(s['is_manifold']):>10}")
print(f"{'─'*70}")
print('Ghi chú: Volume chỉ có khi mesh là watertight (is_manifold=True)')


📊 SUMMARY
──────────────────────────────────────────────────────────────────────
Tên                     Verts    Faces     SA (mm²)    Vol (mm³) Watertight
──────────────────────────────────────────────────────────────────────
Than (Kidney)          20,847   41,706     43,754.5      358,162       True
Khoi U (Tumor)          2,490    5,004      3,074.9        4,919       True
Label 4                 2,315    4,542      2,359.3        1,324       True
Label 5                 1,043    2,078      1,400.0        3,787       True
──────────────────────────────────────────────────────────────────────
Ghi chú: Volume chỉ có khi mesh là watertight (is_manifold=True)


In [18]:
# ============================================================
# CELL 9 — Layout Plotly + Export HTML
# ============================================================
fig.update_layout(
    title=dict(
        text=f'3D Reconstruction — MONAI+PyVista | {case_name}',
        x=0.5,
        font=dict(size=16),
    ),
    scene=dict(
        xaxis=dict(title='X (mm)', showbackground=True, backgroundcolor='#2a2a3e'),
        yaxis=dict(title='Y (mm)', showbackground=True, backgroundcolor='#252535'),
        zaxis=dict(title='Z (mm)', showbackground=True, backgroundcolor='#2a2a3e'),
        aspectmode='data',   # Giữ đúng tỉ lệ vật lý
        bgcolor='#12121f',
        camera=dict(
            eye=dict(x=1.5, y=1.5, z=1.2),
            up=dict(x=0, y=0, z=1),
        ),
    ),
    paper_bgcolor='#12121f',
    font=dict(color='white'),
    legend=dict(
        x=0.01, y=0.99,
        bgcolor='rgba(0,0,0,0.6)',
        font=dict(color='white', size=12),
        bordercolor='rgba(255,255,255,0.2)',
        borderwidth=1,
    ),
    width=1100, height=900,
    margin=dict(r=10, l=10, b=10, t=55),
)

fig.write_html(HTML_OUT, include_plotlyjs='cdn', full_html=True)
size_mb = os.path.getsize(HTML_OUT) / 1e6
print(f'✅ HTML saved : {HTML_OUT}')
print(f'   File size  : {size_mb:.1f} MB')

✅ HTML saved : /kaggle/working/3d_case_00000_merged_monai_pyvista.html
   File size  : 2.3 MB
